In [1]:
import pandas as pd


def load(path):
    """ Load one CPCB continuous-station export averaging of 24H into a table.

    Path : location of the .xlsx export file.

    The export starts with about 10 lines of notes (station, dates, etc.)
    This returns one row per 24-hour window with numeric pollutant columns.
    The code finds the row which says "From Date" and starts the table below it.
    because the number of note lines can change.
    The first 8 columns must be of the order the code assumes and header row
    must contain exactly "From Date" or the code will crash with IndexError.
    """
    raw = pd.read_excel(path, header=None)

    # The number of note lines can change between exports, so find the
    # header row by its text instead of hardcoding its position.
    header_row = raw.index[raw.iloc[:, 0] == "From Date"][0]

    # Keep only the data rows and the 8 columns in use. copy() so the edits
    # below don't affect the raw sheet.
    df = raw.iloc[header_row + 1:, :8].copy()
    df.columns = ["from_date", "to_date", "PM25", "PM10", "NH3", "SO2", "CO", "OZONE"]

    # Readings can load as text so convert to numbers; anything unreadable
    # becomes NaN instead of crashing the load.
    for col in ["PM25", "PM10", "NH3", "SO2", "CO", "OZONE"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df.reset_index(drop=True)


dec = load("chhoti_gwaltoli_2025_12.xlsx")
jun = load("chhoti_gwaltoli_2026_06.xlsx")

# row count and mean PM2.5 / PM10 for each month
for name, d in [("Dec", dec), ("Jun", jun)]:
    print(name, d.shape, d[["PM25", "PM10"]].mean().round(2).to_dict())

Dec (31, 8) {'PM25': 79.47, 'PM10': 143.04}
Jun (29, 8) {'PM25': 23.83, 'PM10': 82.56}


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [ ]:
help(load)

Help on function load in module __main__:

load(path)
    Load one CPCB continuous-station export averaging of 24H into a table.

    Path : location of the .xlsx export file.

    The export starts with about 10 lines of notes (station, dates, etc.)
    This returns one row per 24-hour window with numeric pollutant columns.
    The code finds the row which says "From Date" and starts the table below it.
    because the number of note lines can change.
    The first 8 columns must be of the order the code assumes and header row
    must contain exactly "From Date" or the code will crash with IndexError.

